# 04-2 · Per-diner dish compensation (C4)

| | |
|---|---|
| **What it does** | assigns each member their own best SAFE dish at the chosen restaurant |
| **Proves / tests H4** | per-person dish assignment raises the floor above the restaurant-level MinSat |
| **Source** | original to this paper, ported from `potluck/backend/services/aggregation.py` (`dish_for_each`) |
| **Caution** | the before/after comparison is NOT apples-to-apples — flagged below, not hidden |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## Rebuild the pick from 04-1


In [3]:
# Step 1: rebuild F, S, members, and the fair(0.5) pick, safe dishes attached
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    catalog.append({"option_id": f.stem, "name": f.stem,
                     "menu": [{"name": n, "taste": infer_taste(n)} for n in names]})
for o in catalog:
    nz = [d["taste"] for d in o["menu"] if sum(abs(x) for x in d["taste"]) > 0]
    o["taste"] = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6

F = nbio.build_feasible_set(catalog, members)
S = {o["option_id"]: {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members} for o in F}
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]
labels = {o["option_id"]: o["name"] for o in F}

pick_id, _ = nbio.rank_candidates(candidate_vectors, member_vectors, method="fair", lam=0.5)[0]
pick = next(o for o in F if o["option_id"] == pick_id)
print("pick:", labels[pick_id])


pick: Casa.cafe


## Assign each member their own best safe dish


In [4]:
# Step 2: for each member, the highest-similarity dish from their own safe set
dish_for = {}
for m in members:
    safe = pick["safe_dishes"][m["member_id"]]
    if not safe:
        continue
    best = max(safe, key=lambda d: nbio.taste_similarity(m["taste"], d["taste"]))
    dish_for[m["member_id"]] = {"name": best["name"],
                                 "similarity": nbio.taste_similarity(m["taste"], best["taste"]),
                                 "n_safe_dishes": len(safe)}

nbio.table([{"member": m["display_name"], "s_i on restaurant": S[pick_id][m["member_id"]],
             "their dish": dish_for[m["member_id"]]["name"][:28],
             "dish sim": dish_for[m["member_id"]]["similarity"],
             "safe pool": dish_for[m["member_id"]]["n_safe_dishes"]}
            for m in members if m["member_id"] in dish_for],
           cols=["member", "s_i on restaurant", "their dish", "dish sim", "safe pool"], width=28)


member   s_i on restaurant  their dish                   dish sim  safe pool
----------------------------------------------------------------------------
Vihaan   0.624              Veg Spring Roll              0.955     70       
Vincent  0.746              Veg Spring Roll              0.899     44       
Vasu     0.825              Mocha Milkshake with Coffee  0.933     129      
Alice    0.849              Veg Spring Roll              0.862     129      


## The compensation claim, measured


In [5]:
# Step 3: floor before vs after compensation
restaurant_min = min(S[pick_id][m["member_id"]] for m in members)
dish_min = min(d["similarity"] for d in dish_for.values())
print(f"floor BEFORE (restaurant-level MinSat)  : {restaurant_min:.3f}")
print(f"floor AFTER  (compensated MinSat)        : {dish_min:.3f}")
print(f"change                                    : {dish_min - restaurant_min:+.3f}")


floor BEFORE (restaurant-level MinSat)  : 0.624
floor AFTER  (compensated MinSat)        : 0.862
change                                    : +0.238


> **Caution, stated rather than hidden.** These two numbers are not on the same
> scale. `restaurant_min` is cosine against a restaurant's WHOLE-MENU average
> taste vector; `dish_min` is cosine against ONE dish hand-picked to match a
> single member. A dish chosen to fit one person will almost always score
> higher than any restaurant average — some of the apparent lift is a property
> of the comparison, not of fairness recovered. Report it with this caveat, or
> construct a like-for-like baseline (PRD-5).


### Reading it

The restaurant vote has one winner; the dish assignment has n. Whatever the
scale-mismatch caveat above, the mechanism itself is real: a member who
"loses" the restaurant vote is not stuck with the restaurant's average dish —
they get their own best safe option. **H4** in the paper's plan asks whether
this measurably helps the worst-off member; this notebook is the harness for
that question, honestly caveated.

Next: **05-mediate** — Vihaan's closed-loop mediator and veto token.
